In [62]:
from __future__ import annotations
from datetime import date
from pathlib import Path
from typing import Any
import io
import pandas as pd
import sys
import importlib
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
from models.base import BaseLLM
import models.claude_adapter
importlib.reload(models.claude_adapter)
from schemas import AiEventSearchRequest, EventSearchResponse, AiSearchLLMResponse, QueryParseResult
from services.ai_common import build_user_prompt, load_system_prompt
from services.event_fields import COLUMN_MAP, SEARCH_FIELDS
from services.event_search import _apply_date_filter, _apply_contains_filter, _map_row_to_event

BASE_DIR = ROOT
DEFAULT_PROMPT_PATH = BASE_DIR / "prompts" / "event_ai_search.md"
QUERY_PARSER_PROMPT_PATH = BASE_DIR / "prompts" / "query_parser.md"

In [63]:

# ===================== 쿼리 파싱 =====================

def _parse_query(llm: BaseLLM, query: str) -> QueryParseResult:
    payload = {
        "current_date": date.today().isoformat(),
        "query": query
    }
    system_prompt = load_system_prompt(QUERY_PARSER_PROMPT_PATH)
    user_prompt = build_user_prompt(
        payload,
        task_description="사용자 검색 질의입니다.",
        xml_tag="query_input"
    )
    result = llm.generate(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        response_schema=QueryParseResult
    )
    print(f"[QueryParser] {result.model_dump()}")
    return result

In [64]:
# ===================== CSV 변환 =====================

def _df_to_search_csv(df: pd.DataFrame) -> str:
    kr_to_en = {v: k for k, v in COLUMN_MAP.items() if k in SEARCH_FIELDS}
    available = {kr: en for kr, en in kr_to_en.items() if kr in df.columns}

    search_df = df[list(available.keys())].copy()
    search_df = search_df.rename(columns=available)

    if "event_id" in df.columns:
        search_df.insert(0, "event_id", df["event_id"].values)

    buffer = io.StringIO()
    search_df.to_csv(buffer, index=False)
    return buffer.getvalue()

In [65]:
def _apply_python_filters(df: pd.DataFrame, parsed: QueryParseResult) -> pd.DataFrame:
    filtered = df.copy()

    # 날짜 필터 — 복수 범위 OR 조건
    if parsed.date_ranges:
        start_col = COLUMN_MAP.get("start_date")
        if start_col and start_col in filtered.columns:
            parsed_start = pd.to_datetime(
                filtered[start_col].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0],
                errors='coerce'
            )
            masks = []
            for dr in parsed.date_ranges:
                mask = (
                    (parsed_start >= pd.to_datetime(dr.start_date)) &
                    (parsed_start <= pd.to_datetime(dr.end_date))
                )
                masks.append(mask)

            combined = masks[0]
            for m in masks[1:]:
                combined = combined | m
            filtered = filtered[combined]

    # venue_category 필터
    if parsed.venue_categories:
        vc_col = "_filter_venue_category"
        if vc_col in filtered.columns:
            filtered = _apply_contains_filter(filtered, vc_col, parsed.venue_categories)

    # 주최 필터
    if parsed.organizers:
        org_col = "_filter_organization"
        if org_col in filtered.columns:
            filtered = _apply_contains_filter(filtered, org_col, parsed.organizers)

    # 키워드 필터
    if parsed.keywords:
        kw_col = "_filter_keywords"
        if kw_col in filtered.columns:
            def has_keyword(val: Any) -> bool:
                if not isinstance(val, (list, tuple, set)):
                    return False
                row_kws = {str(v).strip().lower() for v in val if v}
                return any(k.lower() in row_kws for k in parsed.keywords)
            filtered = filtered[filtered[kw_col].apply(has_keyword)]

    # 행사 유형 필터
    if parsed.event_types:
        et_col = "_filter_event_type"
        if et_col in filtered.columns:
            filtered = _apply_contains_filter(filtered, et_col, parsed.event_types)

    # 유료 여부 필터
    if parsed.is_not_free is not None:
        fee_col = COLUMN_MAP.get("is_not_free")
        if fee_col and fee_col in filtered.columns:
            free_indicators = {"아니오", "무료", "n", "없음"}
            paid_indicators = {"예", "유료", "y", "있음"}
            target = paid_indicators if parsed.is_not_free else free_indicators
            filtered = filtered[
                filtered[fee_col].astype(str).str.strip().str.lower().isin(target)
            ]

    print(f"[PythonFilter] {len(df)}개 → {len(filtered)}개")
    for _, row in filtered.iterrows():
        print(f"  - {row.get('제목', '')} | {row.get('시작 일시', '')} | {row.get('장소', '')}")
    return filtered

In [66]:

# ===================== 메타 빌드 =====================

def _build_meta(df: pd.DataFrame) -> dict[str, dict[str, Any]]:
    kr_to_en = {v: k for k, v in COLUMN_MAP.items()}
    meta: dict[str, dict[str, Any]] = {}

    for _, row in df.iterrows():
        event_id = str(row.get("event_id", ""))
        record: dict[str, Any] = {"event_id": event_id}

        for kr_col, en_key in kr_to_en.items():
            raw = row.get(kr_col)
            if en_key in ("keyword", "keywords"):
                record["keywords"] = (
                    [v.strip() for v in str(raw).split(",") if v.strip()]
                    if pd.notna(raw) else []
                )
            else:
                record[en_key] = str(raw) if pd.notna(raw) else ""

        meta[event_id] = record

    return meta


def _assemble_results(matched_ids: list[str], meta: dict[str, dict[str, Any]]):
    return [meta[eid] for eid in matched_ids if eid in meta]


In [67]:
# ===================== 메인 함수 =====================

def ai_search_events(
        parse_llm: BaseLLM,
        search_llm: BaseLLM,
        df: pd.DataFrame,
        request: AiEventSearchRequest,
        *,
        prompt_path: str | Path = DEFAULT_PROMPT_PATH
) -> EventSearchResponse:

    if df.empty:
        return EventSearchResponse(events=[])

    meta = _build_meta(df)

    # 1단계: 쿼리 파싱 (Haiku)
    parsed = _parse_query(parse_llm, request.ai_search)

    # 2단계: Python 필터
    filtered_df = _apply_python_filters(df, parsed)

    if filtered_df.empty:
        print("[AI Search] Python 필터 결과 없음 → 빈 결과 반환")
        return EventSearchResponse(events=[])

    # 3단계: semantic_query 유무로 분기
    if parsed.semantic_query:
        csv_string = _df_to_search_csv(filtered_df)
        payload = {
            "current_date": date.today().isoformat(),
            "query": parsed.semantic_query,
            "events": csv_string
        }
        system_prompt = load_system_prompt(prompt_path)
        user_prompt = build_user_prompt(
            payload,
            task_description="사용자 검색 질의와 필터링된 행사 데이터입니다.",
            xml_tag="ai_search_input"
        )
        llm_response = search_llm.generate(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            response_schema=AiSearchLLMResponse
        )
        unique_ids = list(dict.fromkeys(llm_response.matched_event_ids))
        print(f"[AI Search] semantic 매칭 결과: {len(unique_ids)}건")
        return EventSearchResponse(events=_assemble_results(unique_ids, meta))

    else:
        # Python 필터 결과 그대로 반환
        assembled = [
            _map_row_to_event(row)
            for _, row in filtered_df.iterrows()
        ]
        print(f"[AI Search] Python 필터만으로 결과: {len(assembled)}건")
        return EventSearchResponse(events=assembled)

In [68]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

# DB에서 데이터 로드
from services.event_repository import EventRepository

repository = EventRepository()
df = repository.load_events_dataframe()
print(f"로드된 행사 수: {len(df)}")

[Debug] DB 연결 성공
[debug] DB조회 :    Index                             장소  \
0      1                    Seattle, WA   
1      2     한양대학교 국제관 6층 602호 (디지털강의실)   
2      3                      서울 대한전기협회   
3      4  온라인 (14:00–16:00; 원문 시간대 미기재)   
4      5                            미기재   

                                                  제목  \
0                              U.S. Women in Nuclear   
1                 2026년 iTRS 몬테칼로 이론 및 실무교육(MCNP) 안내   
2                                          일반기계 공인검사   
3  Webinar: NEA–AFCONE Engagement Series with Afr...   
4                              가공배전 교육 (30일/240H) 94   

                                                  주최  \
0                     Nuclear Energy Institute (NEI)   
1                                            한국원자력학회   
2                                       대한전기협회 KEPIC   
3  OECD Nuclear Energy Agency (NEA); African Comm...   
4                                     대한전기협회 전력기술교육원   

                                출처 

In [69]:
from models.factory import load_llm

parse_llm = load_llm(
    provider="claude",
    model="anthropic/claude-haiku-4.5",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

search_llm = load_llm(
    provider="openai",
    model="gpt-5.2",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [70]:
# test_queries = [
#     "올해 겨울 중국에서 열리는 행사",
#     "내년 상반기 무료 AI 컨퍼런스",
#     "실무에 바로 적용할 수 있는 워크숍",
#     "중국에서 열리는 흥미로운 행사",
# ]

test_queries = [
    "2026년에 열리는 행사 알려줘",
    "2026년 국내에서 열리는 행사 알려줘"
]

for query in test_queries:
    print(f"\n{'='*50}")
    print(f"질의: {query}")
    request = AiEventSearchRequest(ai_search=query)
    result = ai_search_events(parse_llm, search_llm, df, request)
    print(f"결과: {len(result.events)}건")
    for e in result.events:  # 상위 3개만 출력
        e_dict = e if isinstance(e, dict) else e.model_dump()
        print(f"  - {e_dict.get('title', '')} | {e_dict.get('start_date', '')} | {e_dict.get('location', '')}")


질의: 2026년에 열리는 행사 알려줘
[Claude] model=anthropic/claude-haiku-4.5, tokens=3964
[QueryParser] {'date_ranges': [{'start_date': '2026-08-13', 'end_date': '2026-12-31'}], 'keywords': [], 'event_types': [], 'organizers': [], 'venue_categories': [], 'is_not_free': None, 'semantic_query': None}
[PythonFilter] 222개 → 175개
  - IEEE PES Webinar: AI for Power System Applications: Hands-on Examples, Selected Applications, and Perspectives – Part 2 | 2026-08-14 00:00:00 KST | 미기재
  - Technical Meeting on Data Needs for Ion Beam Analysis (Hybrid) | 2026-08-24 17:00:00 KST | 미기재
  - 2026 2nd International Conference on Energy Technology and Electrical Engineering (ETEE) | 2026-08-14 (KST; 날짜 전용) | Shenyang, China
  - IEEE PES iGET Webinar: Energy from Space: How Space Solar Power could be a 24/7 firm power source? | 2026-08-15 02:00:00 KST | 미기재
  - 2026 IEEE 10th Forum on Research and Technologies for Society and Industry (RTSI) | 2026-08-16 (KST; 날짜 전용) | Espoo, Finland
  - Global 2026 | 2026-08-16 